# 3 — ASD prevalence metrics by sex and age (OECD cohort)

## Objective

This notebook derives analytical metrics from the processed OECD ASD prevalence dataset.

The goal is to compute structured measures that support the comparison of ASD prevalence by gender and age range across OECD countries, regions and over time.

Main steps:

1. Load processed and auxiliary datasets
2. Merge regional classification data
3. Validate analysis-ready structure
4. Compute sex- and age-based metrics
5. Prepare outputs for downstream analysis and visualization

## 3.1 Environment and setup

This section prepares the execution environment for the analysis.

It includes:
- core library imports
- environment checks

In [9]:
# --- Imports ---

import pandas as pd
import sys

from src.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR, ensure_data_dirs

In [10]:
# --- Environment check ---

print("Python version:", sys.version.split()[0])

try:
    import src
    print("Project package import: OK")
except ImportError:
    print("ERROR: project package 'src' not found")

try:
    import pandas as pd
    print("pandas version:", pd.__version__)
except ImportError:
    print("ERROR: pandas not installed")

Python version: 3.13.5
Project package import: OK
pandas version: 2.2.3


## 3.2 Data ingestion

In this section I load the processed analytical dataset and the auxiliary dataset required for regional classification.

The objective is to locate, validate and load both datasets into the notebook in a controlled and reproducible way.

Key steps:
- locate processed ASD prevalence dataset in the project data directory
- verify that the dataset file exists
- load the dataset into a pandas DataFrame
- locate country-to-region mapping dataset
- verify that the dataset file exists
- load the dataset into a pandas DataFrame

In [11]:
# --- Locate processed ASD prevalence dataset ---

processed_filename = "gbd_asd_prevalence_oecd_processed.csv"

processed_path = PROCESSED_DIR / processed_filename

# --- Verify dataset file exists ---

assert processed_path.exists(), f"Processed dataset not found: {processed_path}"

# --- Load processed dataset ---

df = pd.read_csv(processed_path)

print("Processed dataset located:", processed_path)
print("Shape:", df.shape)

df.head()

Processed dataset located: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\gbd_asd_prevalence_oecd_processed.csv
Shape: (38760, 7)


,country,gender,age_range,year,prevalence,upper_ui,lower_ui
0,Slovenia,Male,<5,1991,939.037586,1855.692963,459.595189
1,Slovenia,Female,<5,1991,363.753608,718.021074,177.685811
2,Slovenia,Male,5-9,1991,920.239063,1984.293942,435.700531
3,Slovenia,Female,5-9,1991,356.527957,772.392177,168.000119
4,Slovenia,Male,10-14,1991,912.201931,1979.073395,358.173398


In [12]:
# --- Locate country-to-region mapping dataset ---

regions_filename = "country_regions_oecd.csv"

regions_path = INTERIM_DIR / regions_filename

# --- Verify dataset file exists ---

assert regions_path.exists(), f"Regions dataset not found: {regions_path}"

# --- Load regions dataset ---

regions = pd.read_csv(regions_path)

print("Regions dataset located:", regions_path)
print("Shape:", regions.shape)

regions.head()

Regions dataset located: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\2_interim\country_regions_oecd.csv
Shape: (38, 2)


,country,region
0,Germany,Europe
1,Australia,Asia-Pacific
2,Austria,Europe
3,Belgium,Europe
4,Canada,North America


In [13]:
# --- Merge processed dataset with country-to-region mapping ---

df = df.merge(regions, on="country", how="left")

print("Dataset shape after merge:", df.shape)
df.head()

Dataset shape after merge: (38760, 8)


,country,gender,age_range,year,prevalence,upper_ui,lower_ui,region
0,Slovenia,Male,<5,1991,939.037586,1855.692963,459.595189,Europe
1,Slovenia,Female,<5,1991,363.753608,718.021074,177.685811,Europe
2,Slovenia,Male,5-9,1991,920.239063,1984.293942,435.700531,Europe
3,Slovenia,Female,5-9,1991,356.527957,772.392177,168.000119,Europe
4,Slovenia,Male,10-14,1991,912.201931,1979.073395,358.173398,Europe


In [14]:
# --- Verify region mapping completeness ---

missing_regions = df["region"].isnull().sum()

print("Missing region values:", missing_regions)
print("Regions:", df["region"].unique())

Missing region values: 0
Regions: ['Europe' 'Middle East' 'Latin America' 'Asia-Pacific' 'North America']


## 3.3 Metrics computation

In this section I derive analytical metrics from the dataset to support the comparison of ASD prevalence by gender and age range.

The objective is to compute structured measures that allow the identification of differences between males and females across age groups, countries, regions and over time.

The analysis is structured across different levels of aggregation:

### 3.3.1 Global metrics (gender × age_range)
- compute average prevalence by gender and age range
- compare prevalence between genders across age groups

### 3.3.2 Temporal trends (gender × year)
- analyze evolution of prevalence over time
- compare trends between genders

### 3.3.3 Country comparison (country × gender)
- evaluate differences across OECD countries
- identify potential outliers

### 3.3.4 Regional comparison (region × gender)
- compare prevalence across geographic regions
- identify macro-level patterns

In [23]:
# --- Compute average prevalence by gender and age range ---

df_gender_age = df.groupby(["gender","age_range"],as_index=False,sort=False).agg({"prevalence":"mean"})

df_gender_age.head()

,gender,age_range,prevalence
0,Male,<5,1511.793289
1,Female,<5,576.661732
2,Male,5-9,1437.185333
3,Female,5-9,537.343805
4,Male,10-14,1370.835216


In [31]:
# Validate structure and completeness of gender-age aggregated dataset

print("Shape:", df_gender_age.shape)

print("\nGender categories:")
print(df_gender_age["gender"].unique())

print("\nAge range categories:")
print(df_gender_age["age_range"].unique())

print("\nMissing values:")
print(df_gender_age.isnull().sum())

print("\nDuplicate rows (gender-age_range):")
print(df_gender_age.duplicated(subset=["gender", "age_range"]).sum())

Shape: (30, 3)

Gender categories:
['Male' 'Female']

Age range categories:
['<5' '5-9' '10-14' '15-19' '20-24' '25-29' '30-34' '35-39' '40-44'
 '45-49' '50-54' '55-59' '60-64' '65-69' '70>']

Missing values:
gender        0
age_range     0
prevalence    0
dtype: int64

Duplicate rows (gender-age_range):
0


In [26]:
# --- Compare prevalence between genders across age groups ---
age_order=["<5","5-9","10-14","15-19","20-24","25-29","30-34","35-39","40-44","45-49","50-54","55-59","60-64","65-69","70>"]

df_gender_age_pivot=(
    df_gender_age
    .pivot(index="age_range",columns="gender",values="prevalence")
    .rename_axis(None,axis=1)
    .reset_index()
)

# Reapply categorical order after pivot
df_gender_age_pivot["age_range"]=pd.Categorical(df_gender_age_pivot["age_range"],categories=age_order,ordered=True)
df_gender_age_pivot=df_gender_age_pivot.sort_values("age_range")

# Compute gender gap metrics
df_gender_age_pivot["difference"]=df_gender_age_pivot["Male"]-df_gender_age_pivot["Female"]
df_gender_age_pivot["ratio"]=df_gender_age_pivot["Male"]/df_gender_age_pivot["Female"]

# Convert to percentage (from rate per 100,000)
df_gender_age_pivot["Female_pct"]=df_gender_age_pivot["Female"]/1000
df_gender_age_pivot["Male_pct"]=df_gender_age_pivot["Male"]/1000

# Rounding for clarity
df_gender_age_pivot["Female"]=df_gender_age_pivot["Female"].round(0)
df_gender_age_pivot["Male"]=df_gender_age_pivot["Male"].round(0)
df_gender_age_pivot["difference"]=df_gender_age_pivot["difference"].round(0)
df_gender_age_pivot["ratio"]=df_gender_age_pivot["ratio"].round(2)
df_gender_age_pivot["Female_pct"]=df_gender_age_pivot["Female_pct"].round(2)
df_gender_age_pivot["Male_pct"]=df_gender_age_pivot["Male_pct"].round(2)

df_gender_age_pivot.head()

,age_range,Female,Male,difference,ratio,Female_pct,Male_pct
14,<5,577.0,1512.0,935.0,2.62,0.58,1.51
8,5-9,537.0,1437.0,900.0,2.67,0.54,1.44
0,10-14,499.0,1371.0,872.0,2.75,0.50,1.37
1,15-19,465.0,1313.0,848.0,2.82,0.46,1.31
2,20-24,436.0,1264.0,829.0,2.90,0.44,1.26


In [32]:
# Validate pivot structure for gender-age comparison

print("Shape:", df_gender_age_pivot.shape)

print("\nColumns:")
print(df_gender_age_pivot.columns)

print("\nMissing values:")
print(df_gender_age_pivot.isnull().sum())

Shape: (15, 7)

Columns:
Index(['age_range', 'Female', 'Male', 'difference', 'ratio', 'Female_pct',
       'Male_pct'],
      dtype='object')

Missing values:
age_range     0
Female        0
Male          0
difference    0
ratio         0
Female_pct    0
Male_pct      0
dtype: int64


In [40]:
# Inspect sample rows to verify metric behavior by gender and age range
df_gender_age_pivot.head()

,age_range,Female,Male,difference,ratio,Female_pct,Male_pct
14,<5,577.0,1512.0,935.0,2.62,0.58,1.51
8,5-9,537.0,1437.0,900.0,2.67,0.54,1.44
0,10-14,499.0,1371.0,872.0,2.75,0.50,1.37
1,15-19,465.0,1313.0,848.0,2.82,0.46,1.31
2,20-24,436.0,1264.0,829.0,2.90,0.44,1.26


### 3.3.2 Temporal trends (gender × year)

In this section I analyze the evolution of ASD prevalence over time.

The objective is to identify temporal trends and compare how prevalence changes between genders across years.

Key steps:
- compute average prevalence by gender and year
- analyze temporal evolution of prevalence
- compare trends between genders

In [29]:
# Compute average ASD prevalence by gender and year to analyze temporal trends across the OECD cohort
df_gender_year = (
    df
    .groupby(["gender", "year"], as_index=False, sort=False)
    .agg(
        prevalence=("prevalence", "mean"),
        upper_ui=("upper_ui", "mean"),
        lower_ui=("lower_ui", "mean")
    )
)

In [30]:
# Validate structure and completeness of gender-year aggregated dataset

print("Shape:", df_gender_year.shape)

print("\nGender categories:")
print(df_gender_year["gender"].unique())

print("\nYear range:")
print(df_gender_year["year"].min(), "-", df_gender_year["year"].max())

print("\nMissing values:")
print(df_gender_year.isnull().sum())

print("\nDuplicate rows (gender-year):")
print(df_gender_year.duplicated(subset=["gender", "year"]).sum())

Shape: (68, 5)

Gender categories:
['Male' 'Female']

Year range:
1990 - 2023

Missing values:
gender        0
year          0
prevalence    0
upper_ui      0
lower_ui      0
dtype: int64

Duplicate rows (gender-year):
0


In [33]:
# Pivot dataset to compare prevalence between genders over time
df_gender_year_pivot = (
    df_gender_year
    .pivot(index="year", columns="gender", values="prevalence")
    .reset_index()
)

In [34]:
# Standardize column names after pivot for consistent metric computation
df_gender_year_pivot.columns.name = None
df_gender_year_pivot = df_gender_year_pivot.rename(
    columns={
        "Female": "female_prevalence",
        "Male": "male_prevalence"
    }
)

In [35]:
# Compute gender comparison metrics over time
df_gender_year_pivot["difference"] = (
    df_gender_year_pivot["female_prevalence"]
    - df_gender_year_pivot["male_prevalence"]
)

df_gender_year_pivot["ratio"] = (
    df_gender_year_pivot["female_prevalence"]
    / df_gender_year_pivot["male_prevalence"]
)

df_gender_year_pivot["female_pct"] = (
    df_gender_year_pivot["female_prevalence"]
    / (
        df_gender_year_pivot["female_prevalence"]
        + df_gender_year_pivot["male_prevalence"]
    )
    * 100
)

df_gender_year_pivot["male_pct"] = (
    df_gender_year_pivot["male_prevalence"]
    / (
        df_gender_year_pivot["female_prevalence"]
        + df_gender_year_pivot["male_prevalence"]
    )
    * 100
)

In [36]:
# Validate structure and completeness of gender-year pivot dataset

print("Shape:", df_gender_year_pivot.shape)

print("\nColumns:")
print(df_gender_year_pivot.columns)

print("\nYear range:")
print(df_gender_year_pivot["year"].min(), "-", df_gender_year_pivot["year"].max())

print("\nMissing values:")
print(df_gender_year_pivot.isnull().sum())

print("\nDuplicate rows (year):")
print(df_gender_year_pivot.duplicated(subset=["year"]).sum())

Shape: (34, 7)

Columns:
Index(['year', 'female_prevalence', 'male_prevalence', 'difference', 'ratio',
       'female_pct', 'male_pct'],
      dtype='object')

Year range:
1990 - 2023

Missing values:
year                 0
female_prevalence    0
male_prevalence      0
difference           0
ratio                0
female_pct           0
male_pct             0
dtype: int64

Duplicate rows (year):
0


In [37]:
# Ensure temporal ordering for consistent trend analysis
df_gender_year_pivot = (
    df_gender_year_pivot
    .sort_values("year")
    .reset_index(drop=True)
)

In [38]:
# Validate final temporal dataset structure and metric consistency

print("Shape:", df_gender_year_pivot.shape)

print("\nYear range:")
print(df_gender_year_pivot["year"].min(), "-", df_gender_year_pivot["year"].max())

print("\nMissing values:")
print(df_gender_year_pivot.isnull().sum())

print("\nDuplicate rows (year):")
print(df_gender_year_pivot.duplicated(subset=["year"]).sum())

Shape: (34, 7)

Year range:
1990 - 2023

Missing values:
year                 0
female_prevalence    0
male_prevalence      0
difference           0
ratio                0
female_pct           0
male_pct             0
dtype: int64

Duplicate rows (year):
0


In [39]:
# Inspect sample rows to verify metric behavior over time
df_gender_year_pivot.head()

,year,female_prevalence,male_prevalence,difference,ratio,female_pct,male_pct
0,1990,337.257827,1062.936598,-725.678770,0.317289,24.086500,75.913500
1,1991,338.550041,1064.618783,-726.068743,0.318001,24.127534,75.872466
2,1992,339.970506,1066.683471,-726.712965,0.318717,24.168737,75.831263
3,1993,341.566712,1069.089177,-727.522465,0.319493,24.213326,75.786674
4,1994,343.340101,1071.897531,-728.557430,0.320311,24.260244,75.739756
